# ITBIS — Insider Threat Behavioral Intelligence System
## Milestone 2: Unsupervised Anomaly Detection using Isolation Forest & Security Alert Generation

**Notebook Overview:**
- **Dataset**: 10,000 synthetically seeded multi-indicator telemetry activity logs (`data/synthetic_activity_logs_10000.csv`).
- **Model**: `sklearn.ensemble.IsolationForest` (unsupervised ensemble anomaly detector).
- **Objective**: Learn normal behavioral baselines per employee across 6 indicators, detect subtle multi-dimensional outlier combinations, and generate automated high-fidelity security alerts with recommended remediation actions.

### Step 1: Import Dependencies and Configure Environment

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Set plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print('Libraries imported successfully!')

### Step 2: Load the 10,000 Synthetic Telemetry Activity Logs Dataset

In [ ]:
# Locate and load the 10,000 synthetic activity logs CSV
csv_path = 'data/synthetic_activity_logs_10000.csv'
if not os.path.exists(csv_path):
    csv_path = '../data/synthetic_activity_logs_10000.csv'

df_logs = pd.read_csv(csv_path)
df_logs['timestamp'] = pd.to_datetime(df_logs['timestamp'])

print(f'Loaded {len(df_logs):,} activity logs successfully!')
print('Dataset Shape:', df_logs.shape)
df_logs.head()

### Step 3: Exploratory Data Analysis (EDA) on 10,000 Data Points

In [ ]:
# 1. Event Types Breakdown
print('--- Event Types Distribution ---')
event_counts = df_logs['event_type'].value_counts()
print(event_counts)

# 2. Activity count per employee
print('\n--- Activity Distribution Across Monitored Employees ---')
emp_activity = df_logs.groupby(['employee_id', 'employee_name', 'department']).size().reset_index(name='total_events')
print(emp_activity.to_string(index=False))

# Visualizing event distribution
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
event_counts.plot(kind='barh', ax=ax[0], color='#004ac6', edgecolor='black')
ax[0].set_title('Distribution of 10,000 Telemetry Event Types')
ax[0].set_xlabel('Count')

df_logs['decimal_hour'].plot(kind='hist', bins=24, ax=ax[1], color='#1e40af', edgecolor='black')
ax[1].set_title('Hourly Telemetry Activity Distribution (0h - 24h)')
ax[1].set_xlabel('Decimal Hour of Day')
plt.tight_layout()
plt.show()

### Step 4: Multi-Indicator Behavioral Feature Engineering
Here we aggregate the raw 10,000 time-series events into high-dimensional behavioral feature vectors for each employee covering the 6 indicators:
1. **Login Times**: Average login hour, standard deviation spread, and off-hours login count (10 PM – 5 AM).
2. **Resource Access Frequency**: Daily file downloads, uploads, and remote sessions.
3. **Device Usage**: Unique devices utilized and unknown client count.
4. **Application Usage**: Privilege command attempts and denied administrative actions.
5. **Data Transfer Volume**: Total egress MB, peak single transfer MB, and mass USB egress volume.
6. **Communication Patterns**: Daily email frequencies and total attachment volumes.

In [ ]:
# Feature Engineering per Employee
feature_rows = []

for emp_id, group in df_logs.groupby('employee_id'):
    emp_name = group['employee_name'].iloc[0]
    dept = group['department'].iloc[0]
    designation = group['designation'].iloc[0]
    
    # 1. Login metrics
    logins = group[group['event_type'] == 'login']
    login_hours = logins['decimal_hour'].dropna()
    avg_login = float(login_hours.mean()) if len(login_hours) > 0 else 9.0
    std_login = float(login_hours.std()) if len(login_hours) > 1 else 0.5
    off_hours_logins = int(((login_hours < 6.0) | (login_hours > 22.0)).sum())
    
    # 2. Resource access metrics
    access_events = group[group['event_type'].isin(['file_download', 'file_upload', 'remote_access', 'usb_connect', 'data_transfer'])]
    daily_accesses = access_events.groupby(access_events['timestamp'].dt.date).size()
    mean_daily_access = float(daily_accesses.mean()) if len(daily_accesses) > 0 else 5.0
    std_daily_access = float(daily_accesses.std()) if len(daily_accesses) > 1 else 1.0
    
    # 3. Data Transfer & USB metrics
    total_transfer_mb = float(group['transferred_mb'].sum())
    avg_transfer_mb = float(group['transferred_mb'].mean())
    max_single_transfer_mb = float(group['transferred_mb'].max())
    
    usb_events = group[group['event_type'] == 'usb_connect']
    usb_count = len(usb_events)
    usb_total_mb = float(usb_events['transferred_mb'].sum())
    
    # 4. Privilege & Security Flags
    priv_events = group[group['event_type'] == 'privilege_change']
    sudo_attempts = int(priv_events['command'].str.contains('sudo|root|admin', case=False, na=False).sum())
    denied_events = int((group['status'].str.contains('denied|failed', case=False, na=False)).sum())
    critical_flags = int((group['risk_flag'].str.contains('critical|exfiltration|unauthorized', case=False, na=False)).sum())
    
    # 5. Device diversity
    unique_devices = int(group['device'].replace('', np.nan).nunique())
    
    # 6. Communication patterns
    emails = group[group['event_type'] == 'email_activity']
    email_count = len(emails)
    
    feature_rows.append({
        'employee_id': emp_id,
        'name': emp_name,
        'department': dept,
        'designation': designation,
        'avg_login_hour': round(avg_login, 2),
        'std_login_hour': round(std_login, 2),
        'off_hours_logins': off_hours_logins,
        'mean_daily_access': round(mean_daily_access, 2),
        'std_daily_access': round(std_daily_access, 2),
        'total_transfer_mb': round(total_transfer_mb, 2),
        'avg_transfer_mb': round(avg_transfer_mb, 2),
        'max_single_transfer_mb': round(max_single_transfer_mb, 2),
        'usb_event_count': usb_count,
        'usb_total_mb': round(usb_total_mb, 2),
        'sudo_attempts': sudo_attempts,
        'denied_events': denied_events,
        'critical_flags': critical_flags,
        'unique_devices': unique_devices,
        'email_count': email_count
    })

df_features = pd.DataFrame(feature_rows)
print(f'Engineered feature matrix for {len(df_features)} monitored employees.')
df_features

### Step 5: Train Unsupervised Isolation Forest Anomaly Detection Model

**Isolation Forest Theory:**
- Works by recursively splitting feature space randomly.
- Outliers (threat cases) exist in low-density sparse regions, requiring significantly fewer splits to isolate.
- `contamination=0.15` specifies the anticipated outlier proportion (~15%).

In [ ]:
# Feature columns selection for machine learning
feature_cols = [
    'avg_login_hour',
    'std_login_hour',
    'off_hours_logins',
    'mean_daily_access',
    'std_daily_access',
    'total_transfer_mb',
    'avg_transfer_mb',
    'max_single_transfer_mb',
    'usb_event_count',
    'usb_total_mb',
    'sudo_attempts',
    'denied_events',
    'critical_flags',
    'unique_devices',
    'email_count'
]

X = df_features[feature_cols].copy()

# Standardize features for uniform scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Initialize & Fit Isolation Forest
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.15,
    max_samples='auto',
    random_state=42
)
iso_forest.fit(X_scaled)

print('✓ Isolation Forest Model fitted successfully!')

### Step 6: Anomaly Detection & Threat Outlier Scoring

In [ ]:
# Compute decision scores and outlier predictions
# decision_function: lower/more negative values indicate higher abnormality
df_features['anomaly_score'] = np.round(iso_forest.decision_function(X_scaled), 4)
# predict: -1 for anomalies / outliers, +1 for inliers / normal behavior
df_features['is_outlier'] = iso_forest.predict(X_scaled) == -1

# Rank by threat severity (Rank 1 = most anomalous)
df_features['threat_rank'] = df_features['anomaly_score'].rank(ascending=True, method='min').astype(int)

# Sort by highest threat rank
df_results = df_features.sort_values(by='anomaly_score', ascending=True).reset_index(drop=True)

# Assign Risk Tier
def assign_risk_tier(row):
    if row['anomaly_score'] < -0.05 or row['critical_flags'] > 10:
        return 'Critical'
    elif row['is_outlier'] or row['critical_flags'] > 0:
        return 'High'
    elif row['anomaly_score'] < 0.05:
        return 'Medium'
    else:
        return 'Low'

df_results['risk_tier'] = df_results.apply(assign_risk_tier, axis=1)

print('=== Isolation Forest Detection Results ===')
display_cols = ['threat_rank', 'employee_id', 'name', 'department', 'anomaly_score', 'is_outlier', 'risk_tier', 'usb_total_mb', 'sudo_attempts', 'off_hours_logins']
df_results[display_cols]

### Step 7: Visualizing Isolation Forest Threat Radar & Anomaly Clusters

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Anomaly Scores Bar Chart
colors = ['#dc2626' if out else '#16a34a' for out in df_results['is_outlier']]
ax[0].barh(df_results['name'], df_results['anomaly_score'], color=colors, edgecolor='black')
ax[0].axvline(0, color='black', linestyle='--', alpha=0.7)
ax[0].set_title('Employee ML Anomaly Scores (Negative = Flagged Outliers)')
ax[0].set_xlabel('Isolation Forest Decision Score')

# Plot 2: Scatter of Egress Volume vs Sudo Attempts colored by Anomaly Score
scatter = ax[1].scatter(
    df_results['total_transfer_mb'],
    df_results['sudo_attempts'],
    c=df_results['anomaly_score'],
    cmap='coolwarm_r',
    s=200,
    edgecolors='black',
    linewidths=1.5
)
for _, row in df_results.iterrows():
    if row['is_outlier']:
        ax[1].annotate(f"{row['employee_id']} ({row['name']})", (row['total_transfer_mb'], row['sudo_attempts'] + 0.3), fontweight='bold', color='#b91c1c')

ax[1].set_title('Multi-Factor Threat Matrix (Transfer MB vs Sudo Attempts)')
ax[1].set_xlabel('Total Transfer Volume (MB)')
ax[1].set_ylabel('Privilege Escalation Attempts')
fig.colorbar(scatter, ax=ax[1], label='Anomaly Score')
plt.tight_layout()
plt.show()

### Step 8: Automated Security Alert Generation Engine
Based on the Isolation Forest detections, this pipeline generates structured security incident alerts ready for SOC triage and database ingestion.

In [ ]:
generated_alerts = []
now_iso = datetime.utcnow().isoformat() + 'Z'

for idx, row in df_results.iterrows():
    if not row['is_outlier'] and row['risk_tier'] == 'Low':
        continue
        
    emp_id = row['employee_id']
    emp_name = row['name']
    dept = row['department']
    score = row['anomaly_score']
    risk = row['risk_tier']
    
    # Root Cause Identification
    reasons = []
    rec_actions = []
    
    if row['usb_total_mb'] > 1000:
        reasons.append(f"Mass USB Data Exfiltration ({row['usb_total_mb']:,.1f} MB copied across {row['usb_event_count']} events)")
        rec_actions.append('Immediately revoke USB endpoint storage access and isolate host.')
    if row['sudo_attempts'] > 0:
        reasons.append(f"Unauthorized Privilege Escalation ({row['sudo_attempts']} sudo root execution attempts)")
        rec_actions.append('Lock sudo privileges and review IAM role bindings.')
    if row['off_hours_logins'] > 5:
        reasons.append(f"Anomalous Off-Hours Activity ({row['off_hours_logins']} logins between 10PM-5AM)")
        rec_actions.append('Verify MFA telemetry and confirm on-call shift approval.')
    if row['denied_events'] > 10:
        reasons.append(f"Repeated Access Failures / Brute-force ({row['denied_events']} denied attempts)")
        rec_actions.append('Enforce immediate credential reset and session termination.')
        
    if not reasons:
        reasons.append(f"Multi-indicator statistical deviation flagged by Isolation Forest (score: {score})")
        rec_actions.append('Conduct tier-1 security analyst review on recent activity logs.')
        
    message = f"ML Anomaly: {reasons[0]}"
    details = ' | '.join(reasons)
    recommended = ' '.join(rec_actions)
    
    generated_alerts.append({
        'alert_id': f"ALT-ML-{len(generated_alerts)+1:03d}",
        'employee_id': emp_id,
        'employee_name': emp_name,
        'department': dept,
        'severity': risk,
        'ml_anomaly_score': score,
        'alert_message': message,
        'details': details,
        'recommended_action': recommended,
        'status': 'UNASSIGNED',
        'created_at': now_iso
    })

df_alerts = pd.DataFrame(generated_alerts)
print(f'✓ Generated {len(df_alerts)} Security Alerts from Isolation Forest Detections:')
df_alerts[['alert_id', 'employee_id', 'employee_name', 'severity', 'alert_message', 'recommended_action']]

### Step 9: Export Generated Security Alerts to CSV

In [ ]:
output_csv = 'data/generated_ml_threat_alerts.csv'
df_alerts.to_csv(output_csv, index=False)
print(f'✓ Saved {len(df_alerts)} generated alerts to: {output_csv}')